# Day 4 — Attention & Transformers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [5]:
!pip install -q transformers torch

In [6]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

text = "This project is really interesting and useful."

result = classifier(text)

print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998650550842285}]


## Pre-trained Transformer with Hugging Face

A pre-trained **DistilBERT** model was loaded using the Hugging Face `pipeline` for sentiment analysis.

In [7]:
# !pip install -q kagglehub

In [8]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download("saurabhshahane/fake-news-classification")

print("Dataset path:", path)
print(os.listdir(path))

Using Colab cache for faster access to the 'fake-news-classification' dataset.
Dataset path: /kaggle/input/fake-news-classification
['WELFake_Dataset.csv']


## Load the Dataset

The WELFake dataset was downloaded directly from Kaggle using `kagglehub`.

In [9]:
df = pd.read_csv(os.path.join(path, "WELFake_Dataset.csv"))

print("Shape:", df.shape)
df.head()

Shape: (72134, 4)


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


In [10]:
# Remove unnecessary column
df = df.drop(columns=["Unnamed: 0"])

# Remove missing values
df = df.dropna(subset=["text", "label"])

# Combine title and text
df["content"] = df["title"].fillna("") + " " + df["text"]

X = df["content"]
y = df["label"]

print("Dataset shape:", df.shape)
print(y.value_counts())

Dataset shape: (72095, 4)
label
1    37067
0    35028
Name: count, dtype: int64


## Data Preparation

After cleaning the dataset, **72,095 news articles** remained.

The class distribution was approximately balanced:

| Label | Samples |
|---|---:|
| 0 | 35,028 |
| 1 | 37,067 |

The data was then prepared for the Transformer by combining the `title` and `text` fields into a single input sequence.

The dataset will be split into **training, validation, and test sets** using stratified sampling to preserve the class distribution.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 46140
Validation: 11536
Test: 14419


In [12]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [13]:
train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [14]:
print("Train samples:", len(train_encodings["input_ids"]))
print("Validation samples:", len(val_encodings["input_ids"]))
print("Test samples:", len(test_encodings["input_ids"]))

Train samples: 46140
Validation samples: 11536
Test samples: 14419


from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test.tolist()
})

In [15]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Create Hugging Face datasets
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test.tolist()
})

In [16]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": y_test.tolist()
})

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 46140
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 11536
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14419
})


In [17]:
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print(model.config)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "vocab_size": 30522
}



## Transformer Model

A pretrained **DistilBERT** model was loaded for binary text classification.

The model was configured with **2 output labels** corresponding to the two classes in the WELFake dataset:

- `0` → Fake News
- `1` → Real News

The pretrained DistilBERT weights were successfully loaded. The classification head was newly initialized because the original checkpoint was not trained specifically for fake news classification.

The model is therefore ready to be **fine-tuned on the WELFake training dataset**.

In [18]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./distilbert-welfake",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [19]:
import time

start_time = time.time()

trainer.train()

training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")

Epoch,Training Loss,Validation Loss
1,0.039660,0.034847
2,0.011350,0.030022
3,0.001735,0.040547


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training time: 3549.02 seconds


## Transformer Training

The DistilBERT model was fine-tuned on the WELFake dataset for **3 epochs** using the training and validation sets.

The training and validation losses were:

| Epoch | Training Loss | Validation Loss |
|---|---:|---:|
| 1 | 0.039660 | 0.034847 |
| 2 | 0.011350 | **0.030022** |
| 3 | 0.001735 | 0.040547 |

The lowest validation loss was achieved at **Epoch 2**, so the best model was retained using `load_best_model_at_end=True`.

The training process took approximately **59.15 minutes** on an NVIDIA T4 GPU.

The increase in validation loss during Epoch 3, while training loss continued to decrease, suggests that the model began to show signs of **overfitting**.

The trained Transformer model is now ready for evaluation on the **test set**.

In [20]:
test_results = trainer.evaluate(test_dataset)

print(test_results)

Training Loss,Validation Loss,Epoch
0.001735,0.024650,3


{'eval_loss': 0.02464965172111988}


In [21]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

predictions = trainer.predict(test_dataset)

y_pred = predictions.predictions.argmax(axis=1)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1: {f1:.4f}")

Test Accuracy: 0.9936
Test Precision: 0.9950
Test Recall: 0.9924
Test F1: 0.9937


## Transformer Evaluation

The fine-tuned DistilBERT model was evaluated on the held-out test set.

The model achieved the following results:

| Metric | Score |
|---|---:|
| Test Loss | **0.02465** |
| Test Accuracy | **99.36%** |
| Precision | **99.50%** |
| Recall | **99.24%** |
| F1-score | **99.37%** |

The results show that the Transformer achieved very strong performance on the WELFake test set, with an F1-score of **99.37%**.

### Comparison with Day 3 LSTM

The Transformer was compared with the LSTM model from Day 3 based on test accuracy:

| Model | Dataset | Test Accuracy |
|---|---|---:|
| LSTM | ECG Heartbeat | 89.44% |
| DistilBERT | WELFake | **99.36%** |

The DistilBERT model achieved **9.92 percentage points higher accuracy** than the Day 3 LSTM.

However, this comparison should be interpreted as a high-level comparison because the two models were evaluated on **different datasets and different tasks**. The LSTM classified ECG heartbeat signals, while the Transformer classified fake and real news articles.

Attention allows a model to look at all relevant parts of a sequence at the same time and assign different importance (weights) to each part, while an RNN processes the sequence step by step, carrying information forward through a hidden state as its memory. This makes attention better at directly connecting distant words or features, whereas an RNN's step-by-step memory can make it harder to preserve long-range information.

## Core Model

The **Transformer (DistilBERT)** will serve as the project's core model because it achieved a **99.36% test accuracy** and **99.37% F1-score** on the WELFake dataset, outperforming the Day 3 LSTM's **89.44% test accuracy**. Its attention mechanism also allows it to capture relationships between different parts of the text more effectively than the step-by-step memory of an RNN.

In [22]:

# !pip freeze > requirements.txt